# Feast Banking - Online Feature Retrieval

> **Original source:** [RHRolun/banking-feature-store](https://github.com/RHRolun/banking-feature-store) (branch: `rbac`)  
> **Author:** RHRolun  
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit) -- auto-connects to Feast services on the cluster

**Prerequisites:** Feast FeatureStore CR deployed and Ready (features applied + materialized).

# Banking Feature Store Demo

This notebook demonstrates how to use Feast for banking feature management with:
- Offline feature retrieval for model training
- Online feature retrieval for real-time inference
- Feature services for organized feature access
- Simple on-demand feature transformations


In [ ]:
import os
import subprocess
import pandas as pd
from feast import FeatureStore, RepoConfig
from datetime import datetime, timedelta

NAMESPACE = os.environ.get("NAMESPACE", "")
FEAST_PROJECT = os.environ.get("FEAST_PROJECT", "banking")
FEAST_NAME = os.environ.get("FEAST_NAME", "banking")

if not NAMESPACE:
    try:
        result = subprocess.run(
            ["oc", "project", "-q"], capture_output=True, text=True, timeout=5
        )
        NAMESPACE = result.stdout.strip() if result.returncode == 0 else "a-rh-dept"
    except Exception:
        NAMESPACE = "a-rh-dept"

print(f"Namespace:     {NAMESPACE}")
print(f"Feast Project: {FEAST_PROJECT}")
print(f"Feast Name:    {FEAST_NAME}")


## 1. Initialize Feature Store


In [ ]:
CERT_PATH = "/etc/pki/tls/custom-certs/ca-bundle.crt"
cert = CERT_PATH if os.path.exists(CERT_PATH) else None

SVC_BASE = f"feast-{FEAST_NAME}"

config = RepoConfig(
    project=FEAST_PROJECT,
    provider="local",
    offline_store={
        "type": "remote",
        "host": f"{SVC_BASE}-offline.{NAMESPACE}.svc.cluster.local",
        "port": 443,
        "scheme": "https",
        "cert": cert,
    },
    online_store={
        "type": "remote",
        "path": f"https://{SVC_BASE}-online.{NAMESPACE}.svc.cluster.local:443",
        "cert": cert,
    },
    registry={
        "path": f"{SVC_BASE}-registry.{NAMESPACE}.svc.cluster.local:443",
        "registry_type": "remote",
        "cert": cert,
    },
    auth={"type": "no_auth"},
    entity_key_serialization_version=3,
)

fs = FeatureStore(config=config)
print(f"Feature store initialized (project={FEAST_PROJECT}, namespace={NAMESPACE})")


## 2. Verify Features Are Available

Features were applied and materialized by the Feast operator on the cluster.
If you need to re-materialize, run from the Feast pod:
```
oc exec -n <namespace> deployment/feast-banking -c registry -- feast materialize-incremental $(date -u +'%Y-%m-%dT%H:%M:%S')
```

In [ ]:
feature_views = fs.list_feature_views()
on_demand_views = fs.list_on_demand_feature_views()
feature_services = fs.list_feature_services()

print(f"Feature views:           {len(feature_views)}")
print(f"On-demand feature views: {len(on_demand_views)}")
print(f"Feature services:        {len(feature_services)}")
print("-" * 50)
for fv in feature_views:
    print(f"  {fv.name:40s} entities={[e.name for e in fv.entity_columns]}")
for odfv in on_demand_views:
    print(f"  {odfv.name:40s} (on-demand)")


## 3. List Available Feature Services


In [ ]:
feature_services = fs.list_feature_services()

print("Available Feature Services:")
print("=" * 40)
for service in feature_services:
    print(f"  {service.name}")
    if service.description:
        print(f"    Description: {service.description}")
    projections = getattr(service, "feature_view_projections", None) or getattr(service, "features", [])
    print(f"    Feature views: {len(projections)}")
    print()


## 4. Offline Feature Retrieval (for Model Training)


In [ ]:
print("OFFLINE FEATURE RETRIEVAL (Model Training)")
print("=" * 50)

sample_customers = ["CUST_000001", "CUST_000002", "CUST_000003"]
recent_timestamp = pd.Timestamp('2025-09-15')

entity_df = pd.DataFrame({
    'customer_id': sample_customers,
    'event_timestamp': [recent_timestamp] * len(sample_customers)
})

print(f"Timestamp: {recent_timestamp}")
print(f"Customers: {sample_customers}")

# Use simple_ondemand_risk_service (small) to avoid gRPC metadata size limits
# For large feature services, use individual feature views instead
SERVICE_NAME = "simple_ondemand_risk_service"
print(f"\nUsing '{SERVICE_NAME}' for offline retrieval...")
try:
    training_df = fs.get_historical_features(
        entity_df=entity_df,
        features=fs.get_feature_service(SERVICE_NAME),
        full_feature_names=True
    ).to_df()
    
    print(f"Retrieved {len(training_df)} rows with {len(training_df.columns)} features")
    print("\nSample data:")
    print(training_df.head())
    
except Exception as e:
    print(f"Error: {e}")
    print("Tip: If gRPC metadata size is exceeded, use individual feature views instead of large feature services.")


## 5. Online Feature Retrieval (for Real-time Inference)


In [ ]:
# Online feature retrieval for real-time inference
print("⚡ ONLINE FEATURE RETRIEVAL (Real-time Inference)")
print("=" * 50)

# Use a customer with recent data
customer_id = "CUST_000465"
print(f"🎯 Getting features for customer: {customer_id}")

# Retrieve features using Customer Charter Feature Service
print("\n🎯 Using 'customer_charter_service' for online retrieval...")
try:
    online_features = fs.get_online_features(
        features=fs.get_feature_service("customer_charter_service"),
        entity_rows=[{"customer_id": customer_id}],
        full_feature_names=True
    ).to_dict()
    
    print(f"✅ Retrieved {len(online_features)} features")
    print("\n📊 Feature values:")
    for feature_name, value in online_features.items():
        if feature_name != 'customer_id':
            print(f"   {feature_name}: {value[0]}")
    
except Exception as e:
    print(f"❌ Error: {e}")


## 6. Multiple Feature Services Demo


In [ ]:
# Demonstrate different feature services
print("🎯 MULTIPLE FEATURE SERVICES DEMO")
print("=" * 40)

services_to_demo = [
    "customer_charter_service",
    "call_prediction_service",
    "risk_compliance_service",
]

for service_name in services_to_demo:
    print(f"\n🎯 Using '{service_name}'...")
    try:
        features = fs.get_online_features(
            features=fs.get_feature_service(service_name),
            entity_rows=[{"customer_id": customer_id}],
            full_feature_names=True
        ).to_dict()
        
        print(f"   ✅ Retrieved {len(features)} features")
        # Show first few features
        feature_count = 0
        for feature_name, value in features.items():
            if feature_name != 'customer_id' and feature_count < 3:
                print(f"   {feature_name}: {value[0]}")
                feature_count += 1
        if len(features) > 4:
            print(f"   ... and {len(features) - 4} more features")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")


## 7. Simple On-Demand Feature Transformation


In [ ]:
# Demonstrate the actual Feast on-demand feature view
print("🔄 ON-DEMAND FEATURE VIEW DEMO")
print("=" * 40)

# Get features using the on-demand risk service
print("Using 'simple_ondemand_risk_service' with on-demand transformations...")
try:
    ondemand_features = fs.get_online_features(
        features=fs.get_feature_service("simple_ondemand_risk_service"),
        entity_rows=[{"customer_id": customer_id}],
        full_feature_names=True
    ).to_dict()
    
    print(f"✅ Retrieved {len(ondemand_features)} features")
    print("\n📊 On-demand feature values:")
    for feature_name, value in ondemand_features.items():
        if feature_name != 'customer_id':
            print(f"   {feature_name}: {value[0]}")
    
    risk_key = next((k for k in ondemand_features if 'risk_score' in k.lower()), None)
    if risk_key:
        risk_score = ondemand_features[risk_key][0]
        print(f"\n🎯 On-Demand Risk Score: {risk_score:.1f}/100")
        print(f"   Risk Level: {'LOW' if risk_score > 70 else 'MEDIUM' if risk_score > 40 else 'HIGH'}")
    
except Exception as e:
    print(f"❌ Error: {e}")

print("\n💡 This demonstrates a real Feast on-demand feature view!")
print("   The risk score is calculated in real-time during feature retrieval.")


## 8. Summary


In [ ]:
# Updated Summary with On-Demand Features
print("📊 BANKING FEATURE STORE DEMO SUMMARY")
print("=" * 50)
print("✅ Feature store initialized and configured")
print("✅ Features applied and materialized")
print("✅ Offline feature retrieval for model training")
print("✅ Online feature retrieval for real-time inference")
print("✅ Feature services for organized feature access")
print("✅ On-demand feature view with real-time transformations")
print("\n🎯 Key Benefits:")
print("   • Centralized feature management")
print("   • Consistent feature access across teams")
print("   • Real-time and batch feature serving")
print("   • Feature versioning and lineage")
print("   • Easy feature discovery and reuse")
print("   • Real-time feature transformations")
print("\n🏦 Perfect for banking use cases:")
print("   • Customer risk assessment")
print("   • Fraud detection")
print("   • Call prediction")
print("   • Transaction analysis")
print("   • Customer segmentation")


In [ ]:
print("📊 BANKING FEATURE STORE DEMO SUMMARY")
print("=" * 50)
print("✅ Feature store initialized and configured")
print("✅ Features applied and materialized")
print("✅ Offline feature retrieval for model training")
print("✅ Online feature retrieval for real-time inference")
print("✅ Feature services for organized feature access")
print("✅ On-demand transformations demonstrated")
print("\n🎯 Key Benefits:")
print("   • Centralized feature management")
print("   • Consistent feature access across teams")
print("   • Real-time and batch feature serving")
print("   • Feature versioning and lineage")
print("   • Easy feature discovery and reuse")
print("\n🏦 Perfect for banking use cases:")
print("   • Customer risk assessment")
print("   • Fraud detection")
print("   • Call prediction")
print("   • Transaction analysis")
print("   • Customer segmentation")
